In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
#synthetic database

# Set seed for reproducibility
rng = np.random.default_rng(42)
n_actions = 2300

# 1. Action Metadata & Stratification Tags
rpa_ids = [f"RPA-2026-{i:04d}" for i in range(1, n_actions + 1)]
grades = rng.choice(["GS-11", "GS-12", "GS-13", "GS-14"], size=n_actions, p=[0.20, 0.27, 0.27, 0.26])
authorities = rng.choice(["Delegated Examining", "Merit Promotion", "Direct Hire"], size=n_actions, p=[0.45, 0.35, 0.20])
clearances = rng.choice(["Public Trust", "Secret", "TS/SCI"], size=n_actions, p=[0.50, 0.35, 0.15])
job_series = rng.choice(["0343 - Mgmt Analyst", "2210 - IT Specialist", "1102 - Contracting", "0801 - Engineering"], size=n_actions)

# 2. Initiation Dates (Weighted toward Q1/Q2 federal hiring surges)
start_date = pd.to_datetime("2025-10-01")  # Start of FY26
quarter_weights = np.array([0.32, 0.30, 0.22, 0.16])
quarters = rng.choice([0, 1, 2, 3], size=n_actions, p=quarter_weights)
days_offset = quarters * 90 + rng.integers(0, 90, size=n_actions)
rpa_dates = start_date + pd.to_timedelta(days_offset, unit="D")

# 3. Assign Realistic Conversion Probabilities by Grade & Clearance
is_gs14 = (grades == "GS-14")
is_ts = (clearances == "TS/SCI")
is_mp = (authorities == "Merit Promotion")

# Calibrated so overall blended selections/cert ≈ 0.80 and stick rate ≈ 70%
sel_lambda = np.where(is_gs14, 0.60, 0.87)
stick_prob = np.where(is_gs14, 0.60, 0.74)
stick_prob = np.where(is_ts, stick_prob - 0.12, stick_prob)   # Clearance dropouts
stick_prob = np.where(is_mp, stick_prob + 0.08, stick_prob)   # Internal transfers stick more
stick_prob = np.clip(stick_prob, 0.35, 0.95)

# 4. Simulate Funnel Counts
# Assume 1 cert per RPA for simplicity (or use rng.choice([1, 2], p=[0.8, 0.2]))
certs_issued = np.ones(n_actions, dtype=int)
tjos_made = rng.poisson(lam=sel_lambda)
eods_completed = rng.binomial(n=tjos_made, p=stick_prob)

# 5. Simulate Realistic Stage Cycle Times (in calendar days)
days_to_cert = rng.integers(14, 45, size=n_actions)
days_to_tjo = np.where(tjos_made > 0, rng.integers(15, 50, size=n_actions), np.nan)

# Security clearance heavily impacts TJO-to-EOD duration
base_vetting = np.where(is_ts, rng.integers(90, 210, size=n_actions), rng.integers(25, 75, size=n_actions))
base_vetting = np.where(is_mp, base_vetting * 0.6, base_vetting).astype(int)  # Internal transfers clear faster
days_to_eod = np.where(eods_completed > 0, base_vetting, np.nan)

# Calculate Calendar Dates
cert_dates = rpa_dates + pd.to_timedelta(days_to_cert, unit="D")
tjo_dates = cert_dates + pd.to_timedelta(days_to_tjo, unit="D")
eod_dates = tjo_dates + pd.to_timedelta(days_to_eod, unit="D")

# 6. Assemble DataFrame
df_hr = pd.DataFrame({
    "RPA_ID": rpa_ids,
    "Grade": grades,
    "Hiring_Authority": authorities,
    "Clearance_Required": clearances,
    "Job_Series": job_series,
    "RPA_Initiated_Date": rpa_dates,
    "Cert_Issued_Date": cert_dates,
    "TJO_Accepted_Date": tjo_dates,
    "EOD_Date": eod_dates,
    "Certs_Issued": certs_issued,
    "Selections_TJOs": tjos_made,
    "Hired_EODs": eods_completed,
    "Total_Time_To_Hire_Days": (eod_dates - rpa_dates).days
})

# Export to CSV so it mimics a real HR data drop
df_hr.to_csv("synthetic_federal_hiring_data.csv", index=False)

# Quick Verification Printout
print(f"Total Recruitments:      {len(df_hr)}")
print(f"Blended Sel / Cert:      {df_hr['Selections_TJOs'].sum() / df_hr['Certs_Issued'].sum():.2f}")
print(f"Blended Stick Rate:      {df_hr['Hired_EODs'].sum() / df_hr['Selections_TJOs'].sum():.1%}")
print(f"Total EODs Produced:     {df_hr['Hired_EODs'].sum()}")
print(f"Median Time-to-Hire:     {df_hr['Total_Time_To_Hire_Days'].median():.0f} days\n")
print(df_hr.groupby("Grade")[["Selections_TJOs", "Hired_EODs", "Total_Time_To_Hire_Days"]].mean().round(2))


Total Recruitments:      2300
Blended Sel / Cert:      0.80
Blended Stick Rate:      70.8%
Total EODs Produced:     1306
Median Time-to-Hire:     107 days

       Selections_TJOs  Hired_EODs  Total_Time_To_Hire_Days
Grade                                                      
GS-11             0.82        0.61                   114.79
GS-12             0.91        0.69                   114.81
GS-13             0.89        0.65                   115.66
GS-14             0.60        0.34                   112.63


In [ ]:
total_applicants = 5000
recruitment_requests = 2300
selections_per_cert = 0.8
stick_rate = 0.7

#rng
rng = np.random.default_rng()

#initialize candidates
active = np.ones(total_applicants, dtype=bool) 
certificate_elgible = np.zeroes(total_applicants, dtype=bool) #did not make certificate list
selections_TJO = np.zeroes(total_applicants, dtype=bool)
hired_EOD = np.zeroes(total_applicants, dtype=bool)

while np.any(active):

    total_active = np.sum(active)

    #rng vectors
    eta1 = rng.random(total_applicants)
    eta2 = rng.random(total_applicants)
    eta3 = rng.random(total_applicants)
    




while np.any(active):